# CS2 EXP-4 — NeoBERT-250M + LoRA


## 1. Dependencies


In [1]:
import sys
import subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "--extra-index-url", "https://download.pytorch.org/whl/cu121",
    "torch==2.5.1",
    "torchvision",
    "torchaudio",
    "transformers<4.49.0",  # Avoids the CVE check enforcing PyTorch 2.6
    "peft<0.14.0",
    "accelerate",
    "numpy<2.1.0",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "pyarrow",
    "joblib",
    "tqdm",
    "psutil",
    "einops",   # NeoBERT (chandar-lab/NeoBERT) remote modeling code dependency
], check=True)

# xformers is pinned to match torch==2.5.1 exactly: an unpinned `-U xformers`
# install pulls a newer release that requires a newer torch, silently
# upgrading it out from under this pin. --no-deps stops pip from touching
# torch again to satisfy xformers. flash-attention is not required since
# `use_unpadding=False` in case_study_2/models.py (see PDD sec. 5.2).
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-deps",
    "xformers==0.0.28.post3",
], check=True)

print("Dependencies installed successfully for CUDA 12.1 driver!")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyreft 0.1.0 requires evaluate>=0.4.1, which is not installed.
pyreft 0.1.0 requires gcsfs>=2024.2.0, which is not installed.
pyreft 0.1.0 requires jupyter, which is not installed.
pyreft 0.1.0 requires wandb, which is not installed.
pyreft 0.1.0 requires ydata-profiling>=4.7.0, which is not installed.
pyreft 0.1.0 requires seaborn==0.12.2, but you have seaborn 0.13.2 which is incompatible.
pyreft 0.1.0 requires transformers==4.45.1, but you have transformers 4.48.3 which is incompatible.


Dependencies installed successfully for CUDA 12.1 driver!


## 1.5 Settings


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "Recovery"

WORKSPACE_ROOT = Path.cwd()
REPO_ROOT = WORKSPACE_ROOT / "DiverseVul--IS-Project"
PROJECT_DIR = REPO_ROOT / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

DATA_ROOT = WORKSPACE_ROOT / "IntelligentSystemProject" / "VulnerabilityDetectionData"
PROCESSED_DIR = DATA_ROOT / "processed"
MANIFEST_ROOT = DATA_ROOT / "manifests"
OUTPUT_ROOT = DATA_ROOT / "outputs"

DOWNSAMPLED_PARQUET = PROCESSED_DIR / "rdiversevul_cs1_normalized_plus_abstracted_v2_downsampled20k.parquet"
MANIFEST_PATH = MANIFEST_ROOT / "cs1_shared_rotating_5fold_v1" / "project_grouped_5fold_manifest.parquet"

CODE_COLUMN = "normalized_code"
#CODE_COLUMN = "abstracted_code_v1"
CODE_COLUMN_TAG = "abstracted" if CODE_COLUMN == "abstracted_code_v1" else "normalized"

EXP4_OUTPUT_DIR = OUTPUT_ROOT / "case_study_2" / f"exp4_neobert_lora_v1_{CODE_COLUMN_TAG}"
EXP4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HF_CACHE_DIR = WORKSPACE_ROOT / "IntelligentSystemProject" / "hf_cache"

HF_TOKEN_VALUE = ""
if HF_TOKEN_VALUE:
    os.environ["HF_TOKEN"] = HF_TOKEN_VALUE
else:
    os.environ.pop("HF_TOKEN", None)

RANK = 16
EPOCHS = 10

TRAIN_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2
EVAL_BATCH_SIZE = 64
NUM_WORKERS = 8

STORAGE_CAP_GB = 60

RUN_SMOKE_TEST = True
RUN_OFFICIAL = True

print("Settings loaded.")
print(f"Workspace: {WORKSPACE_ROOT}")
print(f"Repository: {REPO_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Downsampled parquet: {DOWNSAMPLED_PARQUET}")
print(f"Manifest: {MANIFEST_PATH}")
print(f"Hugging Face cache: {HF_CACHE_DIR}")
# (Device is detected and printed in the "Verify GPU, RAM, and storage budget"
# cell below -- DEVICE doesn't exist yet at this point in the notebook.)


## 2. Clone the repository


In [3]:
import urllib.request
import zipfile
from pathlib import Path

if not REPO_ROOT.exists():
    print(f"Downloading repository (branch: {REPO_BRANCH}) without git...")

    clean_url = REPO_URL.removesuffix(".git")
    zip_url = f"{clean_url}/archive/refs/heads/{REPO_BRANCH}.zip"
    zip_path = Path.cwd() / "repo_temp.zip"

    urllib.request.urlretrieve(zip_url, zip_path)

    print("Extracting files...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(Path.cwd())

    repo_name = clean_url.split("/")[-1]
    extracted_folder = Path.cwd() / f"{repo_name}-{REPO_BRANCH}"
    if extracted_folder.exists():
        extracted_folder.rename(REPO_ROOT)

    zip_path.unlink()
    print("Repository setup complete!")
else:
    print(f"Repository already exists at {REPO_ROOT}")


Repository already exists at /workspace/DiverseVul--IS-Project


## 3. Verify GPU, RAM, and storage budget


In [4]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
assert torch.cuda.is_available(), "CUDA is not available!"
assert torch.cuda.device_count() == 1, f"Expected 1 GPU, but PyTorch sees {torch.cuda.device_count()}"
DEVICE = "cuda:0"
print(f"Locked to single GPU: {torch.cuda.get_device_name(0)}")
print(f"Total Visible GPUs in PyTorch: {torch.cuda.device_count()}")

Locked to single GPU: NVIDIA A100-SXM4-80GB
Total Visible GPUs in PyTorch: 1


## 4. Data availability check


In [ ]:
required_data_files = {
    "downsampled parquet": DOWNSAMPLED_PARQUET,
    "shared 5-fold manifest": MANIFEST_PATH,
}

missing = {name: path for name, path in required_data_files.items() if not path.is_file()}

if missing:
    print("Missing required data files:")
    for name, path in missing.items():
        print(f"  - {name}: {path}")
    print(
        "\nThese files are produced by notebooks/scope2_preprocessing.ipynb (downsampling + "
        "manifest-generation sections) and were previously synced through Google Drive. Copy "
        f"them into the paths above, or re-run that notebook. Keep an eye on the {STORAGE_CAP_GB} GB storage cap."
    )
    raise FileNotFoundError("Required processed data/manifest are missing; see instructions above.")
else:
    for name, path in required_data_files.items():
        size_mb = path.stat().st_size / 1e6
        print(f"Found {name}: {path} ({size_mb:.1f} MB)")


## 5. Verify required repository files

In [ ]:
required_repo_files = [
    SRC_DIR / "case_study_2" / "models.py",
    SRC_DIR / "case_study_2" / "data_loader.py",
    SRC_DIR / "case_study_2" / "exp4" / "exp4_lora.py",
    SRC_DIR / "case_study_2" / "exp4" / "exp4_nested_rank.py",
]

missing_repo_files = [str(path) for path in required_repo_files if not path.exists()]
if missing_repo_files:
    raise FileNotFoundError("Required EXP-4 files are missing:\n" + "\n".join(missing_repo_files))

print("Required Case Study 2 files are present.")

## 6. Import project modules


In [ ]:
import sys

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("case_study_2") or mod_name.startswith("case_study_1") or mod_name.startswith("utils"):
        del sys.modules[mod_name]

from case_study_2.models import (
    configure_huggingface_cache,
    load_code_tokenizer,
    DEFAULT_NEOBERT_MODEL,
    DEFAULT_NEOBERT_TOKENIZER,
    count_trainable_parameters,
    CodeSequenceClassifier,
    infer_lora_target_modules,
)
from case_study_2.exp4.exp4_lora import train_lora_model_safe
from case_study_2.exp4.exp4_nested_rank import Exp4Config, run_exp4_nested_rank
from utils import split_manifest
from utils.confidence_intervals import bootstrap_metric_ci, format_ci_report


## 7. Load the downsampled dataset and shared 5-fold manifest

In [ ]:
import pandas as pd

if not DOWNSAMPLED_PARQUET.is_file():
    raise FileNotFoundError(f"Missing downsampled parquet: {DOWNSAMPLED_PARQUET}")
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Missing manifest: {MANIFEST_PATH}")

full_df = pd.read_parquet(DOWNSAMPLED_PARQUET)
manifest_df = split_manifest.load_manifest(MANIFEST_PATH, config=split_manifest.SplitConfig(n_splits=5, random_state=42))

print("full_df rows:", len(full_df))
print("manifest_df rows:", len(manifest_df))

fold_summary_diag = split_manifest.summarize_manifest(
    manifest_df, config=split_manifest.SplitConfig(n_splits=5, random_state=42)
)
display(fold_summary_diag)

fold_size_ratio = fold_summary_diag["test_rows"].max() / fold_summary_diag["test_rows"].min()
print(f"Fold test-size balance: smallest={fold_summary_diag['test_rows'].min()} rows, "
      f"largest={fold_summary_diag['test_rows'].max()} rows, ratio={fold_size_ratio:.2f}x")
if fold_size_ratio > 2.0:
    print("WARNING: fold sizes are notably imbalanced (ratio > 2x) -- "
          "regenerate the manifest via scope2_preprocessing.ipynb with an updated "
          "DOWNSAMPLE_MAX_ROWS_PER_PROJECT.")
else:
    print("Fold sizes look reasonably balanced.")


## 8. Build the dataset frame

In [ ]:
required_columns = {"source_row_id", "normalized_code", "abstracted_code_v1", "label", "project"}
missing_columns = required_columns - set(full_df.columns)
if missing_columns:
    raise ValueError(f"Missing columns in full_df: {missing_columns}")

full_indexed = full_df.set_index("source_row_id", drop=False)
manifest_ids = set(manifest_df["source_row_id"].tolist())
if manifest_ids != set(full_indexed.index):
    raise RuntimeError(
        "Manifest coverage does not match the downsampled dataset exactly. "
        f"Missing={len(set(full_indexed.index) - manifest_ids)}, extra={len(manifest_ids - set(full_indexed.index))}"
    )

dataset_frame = full_indexed.loc[list(manifest_ids)].reset_index(drop=True)
print("dataset_frame rows:", len(dataset_frame))
print("Unique projects:", dataset_frame["project"].nunique())


## 9. Configuration object


In [ ]:
exp4_config = Exp4Config(
    hf_cache_dir=HF_CACHE_DIR,
    code_column=CODE_COLUMN,
    rank=RANK,
    epochs=EPOCHS,
    train_batch_size=TRAIN_BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    num_workers=NUM_WORKERS,
    eval_batch_size=EVAL_BATCH_SIZE,
)

print("Code column:", exp4_config.code_column)
print("rank (fixed):", exp4_config.rank)
print("lora_alpha_multiplier:", exp4_config.lora_alpha_multiplier)
print("learning_rate:", exp4_config.learning_rate)
print("inner_n_splits:", exp4_config.inner_n_splits)


## 10. Sanity-check LoRA target modules


In [16]:
configure_huggingface_cache(HF_CACHE_DIR)
_tok_check = load_code_tokenizer(DEFAULT_NEOBERT_TOKENIZER, hf_cache_dir=HF_CACHE_DIR)

_probe_model = CodeSequenceClassifier(model_name=DEFAULT_NEOBERT_MODEL, freeze_backbone=False, dtype_policy="bfloat16", hf_cache_dir=HF_CACHE_DIR)
_target_modules = infer_lora_target_modules(_probe_model)
print("Inferred LoRA target_modules:", _target_modules)

del _probe_model
torch.cuda.empty_cache()


2026-08-31 18:48:55.003011: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-31 18:48:55.017327: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788202135.033261   20022 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788202135.038012   20022 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-31 18:48:55.055574: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

Inferred LoRA target_modules: ['qkv']


## 11. Smoke test on a small subsample


In [ ]:
import time
from sklearn.metrics import average_precision_score

if RUN_SMOKE_TEST:
    sample_df = dataset_frame.sample(n=min(2000, len(dataset_frame)), random_state=42).reset_index(drop=True)
    smoke_train = sample_df.iloc[:1500].reset_index(drop=True)
    smoke_val = sample_df.iloc[1500:].reset_index(drop=True)

    print(f"[smoke] train={len(smoke_train)} rows | val={len(smoke_val)} rows")

    t0 = time.time()
    smoke_scores, smoke_model, _ = train_lora_model_safe(
        smoke_train, smoke_val, _tok_check, rank=exp4_config.rank,
        lora_alpha=exp4_config.rank * exp4_config.lora_alpha_multiplier,
        learning_rate=exp4_config.learning_rate, epochs=1,
        batch_size=exp4_config.train_batch_size, grad_accum_steps=exp4_config.grad_accum_steps,
        eval_batch_size=exp4_config.eval_batch_size, num_workers=exp4_config.num_workers,
        device=DEVICE, hf_cache_dir=HF_CACHE_DIR, code_column=exp4_config.code_column,
        max_length=exp4_config.max_length,
    )
    smoke_prauc = float(average_precision_score(smoke_val[exp4_config.label_column].values, smoke_scores))
    print(f"[smoke] PR-AUC={smoke_prauc:.4f} | elapsed={(time.time()-t0)/60:.1f} min")

    del smoke_model
    torch.cuda.empty_cache()
else:
    print("RUN_SMOKE_TEST=False; skipping.")


## 12. Official rotating 5-fold run

In [ ]:
if RUN_OFFICIAL:
    results = run_exp4_nested_rank(
        dataset_frame=dataset_frame,
        manifest=manifest_df,
        config=exp4_config,
        output_dir=EXP4_OUTPUT_DIR,
        additional_metadata={
            "input_parquet": str(DOWNSAMPLED_PARQUET),
            "manifest_path": str(MANIFEST_PATH),
        },
    )
    print("Pooled OOF metrics (secondary cross-check):")
    display(pd.DataFrame([{"metric": k, "value": v} for k, v in results["evaluation"]["pooled_metrics"].items()]))
    print("Mean +/- std across the 5 outer folds (headline result):")
    display(results["evaluation"]["fold_summary"])
    print("Selected hyperparameters by outer fold:")
    display(results["selected"])
else:
    results = None
    print("RUN_OFFICIAL=False; skipping.")


## 13. Confidence interval on pooled OOF PR-AUC (ad hoc)

In [ ]:
exp4_oof_ci = bootstrap_metric_ci(
    results["oof_predictions"],
    metric="average_precision_pr_auc",
    n_bootstrap=1000,
    random_state=42,
)
print(format_ci_report(exp4_oof_ci))


## 14. Cleanup

In [ ]:
import gc
import shutil
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("VRAM allocated:", torch.cuda.memory_allocated() / 1e9, "GB")

total, used, free = shutil.disk_usage(WORKSPACE_ROOT)
print(f"Disk usage at {WORKSPACE_ROOT}: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total ({free/1e9:.1f} GB free)")
if used / 1e9 > STORAGE_CAP_GB:
    print(f"WARNING: workspace usage exceeds the {STORAGE_CAP_GB} GB storage cap -- consider pruning old checkpoints under {EXP4_OUTPUT_DIR}.")
